In [1]:
!pip install -q \
transformers==4.41.2 \
datasets==2.19.0 \
accelerate==0.30.1 \
huggingface_hub==0.23.0 \
peft==0.11.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.23.0 which is incompatible.
diffusers 0.38.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggi

In [2]:
import transformers
import datasets
import huggingface_hub
import peft

print("transformers =", transformers.__version__)
print("datasets =", datasets.__version__)
print("hub =", huggingface_hub.__version__)
print("peft =", peft.__version__)

transformers = 4.41.2
datasets = 2.19.0
hub = 0.23.0
peft = 0.11.1


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Loaded Successfully


In [4]:
import pandas as pd

data = {
    "input": [
        "I am stressed about my exams",
        "I feel lonely",
        "I am anxious",
        "I had a bad day",
        "I feel overwhelmed",
        "I am sad",
        "Nobody understands me",
        "I feel nervous about my future"
    ],
    "response": [
        "I understand. Exams can be stressful. Try taking breaks and focusing on one topic at a time.",
        "I'm sorry you're feeling lonely. Would you like to talk about it?",
        "Anxiety can be difficult. Taking slow deep breaths may help.",
        "I'm sorry your day was difficult. Tomorrow is a new opportunity.",
        "It sounds like you have a lot on your mind. Take things one step at a time.",
        "I'm sorry you're feeling sad. Your feelings are important.",
        "That sounds painful. You deserve to be heard and understood.",
        "It's normal to feel uncertain about the future. Focus on what you can control today."
    ]
}

df = pd.DataFrame(data)
df

,input,response
0,I am stressed about my exams,I understand. Exams can be stressful. Try taki...
1,I feel lonely,I'm sorry you're feeling lonely. Would you lik...
2,I am anxious,Anxiety can be difficult. Taking slow deep bre...
3,I had a bad day,I'm sorry your day was difficult. Tomorrow is ...
4,I feel overwhelmed,It sounds like you have a lot on your mind. Ta...
5,I am sad,I'm sorry you're feeling sad. Your feelings ar...
6,Nobody understands me,That sounds painful. You deserve to be heard a...
7,I feel nervous about my future,It's normal to feel uncertain about the future...


In [5]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

print(dataset)

Dataset({
    features: ['input', 'response'],
    num_rows: 8
})


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

print("Model Loaded Successfully")

Model Loaded Successfully


In [7]:
def preprocess(example):

    text = f"User: {example['input']}\nBot: {example['response']}"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

print("Preprocessing Function Ready")

Preprocessing Function Ready


In [8]:
tokenized_dataset = dataset.map(preprocess)

print(tokenized_dataset)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'response', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 8
})


In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./mental_health_bot",
    num_train_epochs=20,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_strategy="no"
)

print("Training Arguments Ready")

Training Arguments Ready


In [13]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("Trainer Ready")

Trainer Ready


In [14]:
trainer.train()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,6.029600
2,3.808700
3,1.207900
4,0.829800
5,0.779100
6,0.847900
7,0.788900
8,0.860100
9,0.706700
10,0.666100


TrainOutput(global_step=80, training_loss=0.3816158816218376, metrics={'train_runtime': 271.5051, 'train_samples_per_second': 0.589, 'train_steps_per_second': 0.295, 'total_flos': 5225935011840.0, 'train_loss': 0.3816158816218376, 'epoch': 20.0})

In [15]:
trainer.save_model("./mental_health_bot")
tokenizer.save_pretrained("./mental_health_bot")

print("Model Saved Successfully")

Model Saved Successfully


In [16]:
import torch

def chat(user_input):

    prompt = f"User: {user_input}\nBot:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_length=80,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

print("Chatbot Ready")

Chatbot Ready


In [17]:
print(chat("I feel lonely"))

User: I feel lonely
Bot: I'm sad about the emotional situation. Would you like to talk about it?


In [18]:
print(chat("I am stressed about exams"))

User: I am stressed about exams
Bot: It's normal to stress about exams. Taking breaks may help.
